In [1]:
import os
import subprocess

def find_matching_files(folder_path):
    """Find and match FGN and CHIMPS files by their identifier."""
    aligned_files = {}
    reference_files = {}

    # Scan the folder for FGN and CHIMPS files
    for filename in os.listdir(folder_path):
        if filename.endswith(".fits"):
            # Extract the identifier (e.g., 018 from FGN_018_12CO_23arcsec.fits or CHIMPS_12CO_G018.fits)
            if filename.endswith("RMS_23arcsec.fits"):
                identifier = filename.split("_")[1][0:3]  # Extract '018' from 'FGN_018_12CO_23arcsec.fits'
                aligned_files[identifier] = filename
           # elif filename.startswith("CHIMPS"): # Original for aligning FGN to CHIMPS
            elif filename.endswith("aligned.fits"): # Modification for aligning FGN rms to FGN data
                #identifier = filename.split("_")[2][1:4]  # Extract '018' from 'CHIMPS_12CO_G018.fits'
                identifier = filename.split("_")[1][0:3]  # Extract '018' from 'FGN_018_12CO_23arcsec.fits'
                reference_files[identifier] = filename

    return aligned_files, reference_files

def align_files(aligned_file, reference_file, low_bound, up_bound, folder_path):
    """Run the wcs align command for the given FGN and CHIMPS files."""
    # Construct the aligned output filename
    fgn_aligned_file = aligned_file.replace(".fits", "_aligned.sdf")
    
    # Build the command
    command = [
        "wcs", "align", "alignref=True", 
        os.path.join(folder_path, aligned_file),
        os.path.join(folder_path, reference_file),
        str(low_bound), str(up_bound),
        os.path.join(folder_path, fgn_aligned_file)
    ]
    
    # Execute the command
    print(f"Running command: {' '.join(command)}")
    subprocess.run(command)

def main(folder_path, low_bound, up_bound):
    # Find matching FGN and CHIMPS files
    aligned_files, reference_files = find_matching_files(folder_path)
    
    # Iterate over the identifiers and run the align command for each pair
    for identifier, aligned_file in aligned_files.items():
        if identifier in reference_files:
            reference_file = reference_files[identifier]
            print(f"Aligning FGN file {aligned_file} with CHIMPS file {reference_file}")
            align_files(aligned_file, reference_file, low_bound, up_bound, folder_path)
        else:
            print(f"No matching CHIMPS file found for FGN identifier {identifier}")

if __name__ == "__main__":
    # Path to the folder containing the FITS files
    folder_path = '/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/Temp13'
    

    # Run the main function
    #main(FOLDER_PATH, LOW_BOUND, UP_BOUND)


In [2]:
def run_command(command, cwd=None):
    """Run a command and print its output."""
    print(f"Running command: {command}")
    result = subprocess.run(command, shell=True, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"Error running command: {result.stderr}")
    else:
        print(result.stdout)

In [3]:
aligned_files, reference_files = find_matching_files(folder_path)

In [5]:
aligned_files
#reference_files

{'040': 'CHIMPS_040_13CO_RMS_23arcsec.fits',
 '038': 'CHIMPS_038_13CO_RMS_23arcsec.fits',
 '032': 'CHIMPS_032_13CO_RMS_23arcsec.fits',
 '044': 'CHIMPS_044_13CO_RMS_23arcsec.fits',
 '036': 'CHIMPS_036_13CO_RMS_23arcsec.fits',
 '042': 'CHIMPS_042_13CO_RMS_23arcsec.fits',
 '037': 'CHIMPS_037_13CO_RMS_23arcsec.fits',
 '030': 'CHIMPS_030_13CO_RMS_23arcsec.fits',
 '046': 'CHIMPS_046_13CO_RMS_23arcsec.fits',
 '034': 'CHIMPS_034_13CO_RMS_23arcsec.fits',
 '026': 'CHIMPS_026_13CO_RMS_23arcsec.fits',
 '022': 'CHIMPS_022_13CO_RMS_23arcsec.fits',
 '028': 'CHIMPS_028_13CO_RMS_23arcsec.fits',
 '024': 'CHIMPS_024_13CO_RMS_23arcsec.fits',
 '018': 'CHIMPS_018_13CO_RMS_23arcsec.fits',
 '020': 'CHIMPS_020_13CO_RMS_23arcsec.fits',
 '027': 'CHIMPS_027_13CO_RMS_23arcsec.fits'}

In [6]:
import subprocess

os.chdir(folder_path)
# Assuming aligned_files and cohrs_files are defined as dictionaries or lists
for identifier in reference_files:
    align_sdf= aligned_files[identifier].replace(".fits", ".sdf") # FUGIN sdf file before alignment
    fgn_aligned_file = aligned_files[identifier].replace(".fits", "_aligned.sdf") # FUGIN sdf file after alignment
    fgn_aligned_fits = fgn_aligned_file.replace(".sdf", ".fits") # FUGIN fits file after alignment
    reference_sdf = reference_files[identifier].replace(".fits", ".sdf") # CHIMPS sdf file
    
    # Prepare the command
    run_command(f"/usr/local/star-2023A/bin/convert/fits2ndf {aligned_files[identifier]} {align_sdf}")
    run_command(f"/usr/local/star-2023A/bin/convert/fits2ndf {reference_files[identifier]} {reference_sdf}")
    command = f"/usr/local/star-2023A/bin/kappa/wcsalign alignref=True {align_sdf}"

    
    # Start the subprocess
    process = subprocess.Popen(command, shell=True, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Prepare the inputs
    inputs = [
        reference_sdf,                  # Reference image
        "",             # Low bound
        "",              # Upper bound
        fgn_aligned_file            # Output file
    ]

    # Write inputs to the process
    for input_value in inputs:
        process.stdin.write(input_value + '\n')
        process.stdin.flush()  # Ensure the input is sent immediately

    # Capture output and error messages
    stdout, stderr = process.communicate()  # Call communicate() before closing stdin

    # Close stdin to signal that no more input will be sent
    process.stdin.close()

    # Print output and errors
    if stdout:
        print(stdout)
    if stderr:
        print(f"Error: {stderr}")

    # Optional: Convert to FITS format
    run_command(f"/usr/local/star-2023A/bin/convert/ndf2fits {fgn_aligned_file} {fgn_aligned_fits}")

    # Print out the matched files for reference
    print(f"Aligned: {aligned_files[identifier]} with {reference_files[identifier]}")

    os.system('rm -rf ' +align_sdf) 
    os.system('rm -rf ' +reference_sdf) 
    os.system('rm -rf ' +fgn_aligned_file) 


Running command: /usr/local/star-2023A/bin/convert/fits2ndf CHIMPS_038_13CO_RMS_23arcsec.fits CHIMPS_038_13CO_RMS_23arcsec.sdf
1 file selected.

Running command: /usr/local/star-2023A/bin/convert/fits2ndf CHIMPS_038_13CO_23arcsec_aligned.fits CHIMPS_038_13CO_23arcsec_aligned.sdf
1 file selected.

REF - The reference image /!/ > CHIMPS_038_13CO_23arcsec_aligned.sdf
LBND - Lower pixel index bounds for output images /[-611,-299,-191]/ > 
UBND - Upper pixel index bounds for output images /[613,301,193]/ > 
OUT - Output images > CHIMPS_038_13CO_RMS_23arcsec_aligned.sdf
  Using sincsinc interpolation kernel.

  Processing CHIMPS_038_13CO_RMS_23arcsec...
    Alignment has occurred within the SKY-DSBSPECTRUM Domain.



Running command: /usr/local/star-2023A/bin/convert/ndf2fits CHIMPS_038_13CO_RMS_23arcsec_aligned.sdf CHIMPS_038_13CO_RMS_23arcsec_aligned.fits
1 NDF selected.
The 1st NDF (CHIMPS_038_13CO_RMS_23arcsec_aligned) does not have the selected
QUALITY component.
The output FITS file wi

In [7]:
for identifier in aligned_files:
    print(aligned_files[identifier])

CHIMPS_040_13CO_RMS_23arcsec.fits
CHIMPS_038_13CO_RMS_23arcsec.fits
CHIMPS_032_13CO_RMS_23arcsec.fits
CHIMPS_044_13CO_RMS_23arcsec.fits
CHIMPS_036_13CO_RMS_23arcsec.fits
CHIMPS_042_13CO_RMS_23arcsec.fits
CHIMPS_037_13CO_RMS_23arcsec.fits
CHIMPS_030_13CO_RMS_23arcsec.fits
CHIMPS_046_13CO_RMS_23arcsec.fits
CHIMPS_034_13CO_RMS_23arcsec.fits
CHIMPS_026_13CO_RMS_23arcsec.fits
CHIMPS_022_13CO_RMS_23arcsec.fits
CHIMPS_028_13CO_RMS_23arcsec.fits
CHIMPS_024_13CO_RMS_23arcsec.fits
CHIMPS_018_13CO_RMS_23arcsec.fits
CHIMPS_020_13CO_RMS_23arcsec.fits
CHIMPS_027_13CO_RMS_23arcsec.fits
